# Week 4 Homework: Predicting Diabetes with BRFSS Data

## Purpose of Homework

This homework assignment will help you practice building and evaluating a logistic regression model for classification. You will predict **whether someone has diabetes** using health status, health behaviors, healthcare access, and demographic data from the Behavioral Risk Factor Surveillance System (BRFSS).

You are encouraged to refer to lecture content and liberally use course resources such as the discussion board and office hours.

## Logistics

Due date: The homework is due 12:00pm on Thursday, February 5, 2026.

You will submit your homework on [MarkUs](https://markus.teach.cs.toronto.edu/markus/). 

1. Download this file (`STA272_hw4_student.ipynb`) from JupyterHub. (See [our JupyterHub Guide](../guides/jupyterhub_guide.ipynb) for detailed instructions.)
2. Submit this file to MarkUs under the hw4 assignment. (See [our MarkUs Guide](../guides/markus_guide.ipynb) for detailed instructions.)

All homeworks will take place in a Jupyter notebook (like this one). When you are done, you will download this notebook and submit it to MarkUs.

## About the Data

The **Behavioral Risk Factor Surveillance System (BRFSS)** is an annual health survey conducted by the CDC. It collects data on health-related risk behaviors, chronic health conditions, and use of preventive services from over 400,000 U.S. adults.

We will use a random sample of 75,000 observations from the 2024 BRFSS survey. The dataset contains many variables with cryptic labeling and coding of data and missing values. Your task is to select relevant variables, clean them, and build a model to predict diabetes status.

## Data Dictionary

Here are the variables you will work with:

### Outcome Variable
| Code | Description | Values |
|------|-------------|--------|
| `DIABETE4` | Diabetes status | 1=Yes, 2=Yes (pregnancy only), 3=No, 4=Pre-diabetes |

### Health Status
| Code | Description | Values |
|------|-------------|--------|
| `GENHLTH` | General health status | 1=Excellent, 2=Very good, 3=Good, 4=Fair, 5=Poor |
| `PHYSHLTH` | Days physical health not good (past 30 days) | 1-30, 88=None |
| `CVDINFR4` | Ever told you had a heart attack (MI) | 1=Yes, 2=No |
| `CVDCRHD4` | Ever told you had coronary heart disease | 1=Yes, 2=No |
| `HAVARTH4` | Ever told you had arthritis | 1=Yes, 2=No |

### Health Behaviors
| Code | Description | Values |
|------|-------------|--------|
| `SMOKE100` | Smoked 100+ cigarettes in life | 1=Yes, 2=No |
| `EXERANY2` | Exercise in past 30 days | 1=Yes, 2=No |
| `_BMI5` | BMI (multiply by 0.01 for actual value) | e.g., 2500 = 25.00 |

### Healthcare Access
| Code | Description | Values |
|------|-------------|--------|
| `PRIMINS2` | Primary source of health insurance | 1-10 (various types), 88=None |
| `MEDCOST1` | Could not see doctor due to cost (past 12 mo) | 1=Yes, 2=No |
| `CHECKUP1` | Last routine checkup | 1=Past year, 2=1-2 years, 3=2-5 years, 4=5+ years, 8=Never |

### Demographics
| Code | Description | Values |
|------|-------------|--------|
| `_AGE80` | Age (top-coded at 80) | 18-80 |
| `_SEX` | Sex | 1=Male, 2=Female |
| `EDUCA` | Education level | 1-6 (1=Never attended, 6=College graduate) |
| `INCOME3` | Income level | 1-11 (increasing income) |

### Missing Value Codes
- **7, 77, 777, 7777**: Don't know / Not sure
- **9, 99, 999, 9999**: Refused
- **BLANK/NaN**: Not asked or missing

## Task #1: Load the Data

Read `brfss_sample_raw.csv` into a pandas dataframe called `brfss`.

In [ ]:
# Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.api as sm
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import roc_curve, roc_auc_score

# Read the CSV file into a dataframe
brfss = pd.read_csv('brfss_sample_raw.csv')

# Display the shape and first few rows
print(f"Dataset shape: {brfss.shape}")
print(f"Number of variables: {brfss.shape[1]}")
brfss.head()

## Task #2: Select Variables

Using the data dictionary above, select the variables you need for analysis. Create a list called `selected_vars` containing the variable codes, then subset the dataframe to only include these columns.


In [ ]:
# Select variables from the data dictionary
# TODO: Fill in the list of variable codes you want to include
selected_vars = [
    # Outcome
    '...',     # Diabetes status
    # Health Status
    '...',     # General health
    '...',     # Days physical health not good
    '...',     # Ever had heart attack
    '...',     # Coronary heart disease
    '...',     # Arthritis
    # Health Behaviors
    '...',     # Ever smoked
    '...',     # Exercise
    '...',     # BMI
    # Healthcare Access
    '...',     # Primary insurance
    '...',     # Could not afford doctor
    '...',     # Last checkup
    # Demographics
    '...',     # Age
    '...',     # Sex
    '...',     # Education
]

# Check which variables exist in the dataset
available_vars = [v for v in selected_vars if v in brfss.columns]
missing_vars = [v for v in selected_vars if v not in brfss.columns]
if missing_vars:
    print(f"Warning: Variables not found in dataset: {missing_vars}")

# Subset the dataframe
df = brfss[available_vars].copy()
print(f"\nSubset shape: {df.shape}")
df.head()

## Task #3: Clean Missing Values

BRFSS uses special codes for missing values (7, 9, 77, 99, etc. for "Don't know" or "Refused"). Replace these codes with `NaN`.

Also handle the special case where `PHYSHLTH` uses 88 to mean "None" (0 days), and `CHECKUP1` uses 8 to mean "Never" (which we'll treat as a valid response, not missing).

**Hint:** Create a set of missing codes and use `.isin()` to identify them.

In [ ]:
# Replace BRFSS missing codes with NaN
# TODO: Create a set containing the missing value codes (7, 9, 77, 99, etc.)
missing_codes = {...}

# TODO: Loop through columns and replace missing codes with np.nan
for col in df.columns:
    ...

# Special handling for PHYSHLTH: 88 means "None" (0 days)
# TODO: Replace 88 with 0 in the PHYSHLTH column
if 'PHYSHLTH' in df.columns:
    ...

print("Missing values after cleaning:")
print(df.isnull().sum())

## Task #4: Rename Variables

Rename the cryptic BRFSS codes to meaningful names using a dictionary and `.rename()`. Some have been done for you but please fill in the last three.

Also convert `_BMI5` to actual BMI values by dividing `bmi_raw` by 100 to create a variable named `bmi`.

In [ ]:
# Rename columns to meaningful names
# TODO: Complete the rename dictionary - fill in the last 3 entries
rename_dict = {
    'DIABETE4': 'diabetes',
    'GENHLTH': 'general_health',
    'PHYSHLTH': 'days_phys_bad',
    'CVDINFR4': 'heart_attack',
    'CVDCRHD4': 'heart_disease',
    'HAVARTH4': 'arthritis',
    'SMOKE100': 'ever_smoked',
    'EXERANY2': 'exercises',
    '_BMI5': 'bmi_raw',
    'PRIMINS2': 'insurance_type',
    'MEDCOST1': 'cost_barrier',
    'CHECKUP1': 'last_checkup',
    '_AGE80': '...',      # TODO: fill in
    '_SEX': '...',        # TODO: fill in
    'EDUCA': '...'        # TODO: fill in
}

df = df.rename(columns={k: v for k, v in rename_dict.items() if k in df.columns})

# Convert BMI (divide by 100 to get actual value)
# TODO: Create a new column 'bmi' by dividing 'bmi_raw' by 100, then drop 'bmi_raw'
if 'bmi_raw' in df.columns:
    df['bmi'] = ...
    df = df.drop(columns=['bmi_raw'])

print("Renamed columns:")
print(df.columns.tolist())
df.describe()

## Task #5: Create Binary Variables

Create a binary outcome variable called `has_diabetes` that equals 1 if the person has diabetes (original code = 1), and 0 otherwise (codes 2, 3, 4).

Also create binary versions of other categorical variables:
- `ever_smoked_binary`: 1 if ever smoked, 0 otherwise
- `exercises_binary`: 1 if exercises, 0 otherwise
- `had_heart_attack`: 1 if had heart attack (code = 1), 0 otherwise
- `has_heart_disease`: 1 if has coronary heart disease (code = 1), 0 otherwise
- `has_arthritis`: 1 if has arthritis (code = 1), 0 otherwise
- `is_male`: 1 if male, 0 if female
- `has_insurance`: 1 if has any insurance (codes 1-10), 0 if none (code = 88)
- `has_cost_barrier`: 1 if had cost barrier to care, 0 otherwise

In [ ]:
# Create binary outcome: 1 = has diabetes, 0 = no diabetes
# Original: 1=Yes, 2=Yes (pregnancy), 3=No, 4=Pre-diabetes
# TODO: Create has_diabetes column (1 if diabetes==1, 0 otherwise)
df['has_diabetes'] = ...
df.loc[df['diabetes'].isna(), 'has_diabetes'] = np.nan

# Ever smoked: 1=Yes, 2=No -> 1=Yes, 0=No
# TODO: Create ever_smoked_binary column
if 'ever_smoked' in df.columns:
    df['ever_smoked_binary'] = ...
    df.loc[df['ever_smoked'].isna(), 'ever_smoked_binary'] = np.nan

# Exercises: 1=Yes, 2=No -> 1=Yes, 0=No
# TODO: Create exercises_binary column
if 'exercises' in df.columns:
    df['exercises_binary'] = ...
    df.loc[df['exercises'].isna(), 'exercises_binary'] = np.nan

# Heart attack: 1=Yes, 2=No -> 1=Yes, 0=No
if 'heart_attack' in df.columns:
    df['had_heart_attack'] = (df['heart_attack'] == 1).astype(float)
    df.loc[df['heart_attack'].isna(), 'had_heart_attack'] = np.nan

# Coronary heart disease: 1=Yes, 2=No -> 1=Yes, 0=No
if 'heart_disease' in df.columns:
    df['has_heart_disease'] = (df['heart_disease'] == 1).astype(float)
    df.loc[df['heart_disease'].isna(), 'has_heart_disease'] = np.nan

# Arthritis: 1=Yes, 2=No -> 1=Yes, 0=No
if 'arthritis' in df.columns:
    df['has_arthritis'] = (df['arthritis'] == 1).astype(float)
    df.loc[df['arthritis'].isna(), 'has_arthritis'] = np.nan

# Sex: 1=Male, 2=Female -> 1=Male, 0=Female
# TODO: Create is_male column
if 'sex' in df.columns:
    df['is_male'] = ...
    df.loc[df['sex'].isna(), 'is_male'] = np.nan

# Has insurance: 1-10 = various types, 88 = None -> 1=Has insurance, 0=None
if 'insurance_type' in df.columns:
    df['has_insurance'] = ((df['insurance_type'] >= 1) & (df['insurance_type'] <= 10)).astype(float)
    df.loc[df['insurance_type'].isna(), 'has_insurance'] = np.nan

# Cost barrier: 1=Yes, 2=No -> 1=Yes, 0=No
if 'cost_barrier' in df.columns:
    df['has_cost_barrier'] = (df['cost_barrier'] == 1).astype(float)
    df.loc[df['cost_barrier'].isna(), 'has_cost_barrier'] = np.nan

print("Distribution of diabetes (binary):")
print(df['has_diabetes'].value_counts(dropna=False))
print(f"\nPercentage with diabetes: {df['has_diabetes'].mean()*100:.1f}%")

## Task #6: Visualize the Outcome

Make a bar plot showing the distribution of the outcome variable (`has_diabetes`).

In [ ]:
# Bar plot of outcome distribution
# TODO: Get value counts of has_diabetes, reindexed to show [0, 1]
counts = ...

# TODO: Create a bar plot
ax = counts.plot(kind='bar', color=["#228ab7", "#e74c3c"])
ax.set_xticklabels(['No Diabetes (0)', 'Diabetes (1)'], rotation=0)
plt.xlabel('Diabetes Status')
plt.ylabel('Count')
plt.title('Distribution of Diabetes Status')
plt.show()

## Task #7: Explore Predictors

Create boxplots comparing `bmi` and `age` between those with and without diabetes.

In [ ]:
# Boxplot: BMI and Age by diabetes status
# TODO: Create a figure with 2 subplots side by side
fig, axes = plt.subplots(...)

# TODO: Create boxplot for BMI by diabetes status
df.boxplot(column='...', by='...', ax=axes[0])
axes[0].set_xlabel('Diabetes Status')
axes[0].set_ylabel('BMI')
axes[0].set_title('BMI by Diabetes Status')

# TODO: Create boxplot for Age by diabetes status
df.boxplot(column='...', by='...', ax=axes[1])
axes[1].set_xlabel('Diabetes Status')
axes[1].set_ylabel('Age')
axes[1].set_title('Age by Diabetes Status')

plt.suptitle('')
plt.tight_layout()
plt.show()

## Task #8: Prepare Data for Modeling

Select features for the model, check for missing values, and remove rows with missing data in key variables. Store the result in `df_clean`.

In [ ]:
# Select features for the model
# TODO: Fill in the feature column names
feature_cols = [
    # Demographics
    '...',
    '...',
    '...',
    # Health Status
    '...',
    '...',
    '...',
    '...',
    # Health Behaviors
    '...',
    '...',
    '...',
    # Healthcare Access
    '...',
]

# Keep only features that exist
feature_cols = [c for c in feature_cols if c in df.columns]
print(f"Features for model: {feature_cols}")

# Create analysis dataset
# TODO: Create list of analysis variables (features + outcome)
analysis_vars = feature_cols + ['has_diabetes']
df_clean = df[analysis_vars].dropna()

print(f"\nComplete cases: {len(df_clean)} out of {len(df)} ({100*len(df_clean)/len(df):.1f}%)")

In [ ]:
# Define X and y
# TODO: Set X to be the features and y to be the outcome
X = df_clean[...]
y = df_clean[...]

# Train/test split
# TODO: Use train_test_split with test_size=0.2, random_state=42, and stratify=y
X_train, X_test, y_train, y_test = train_test_split(
    ..., ..., test_size=..., random_state=..., stratify=...
)

print(f"Training set: {len(X_train)} observations")
print(f"Test set: {len(X_test)} observations")
print(f"\nClass distribution in training set:")
print(y_train.value_counts(normalize=True))

## Task #9: Fit Logistic Regression

Fit a logistic regression model using `statsmodels` on the training data. Display the model summary.

**Hint:** Remember to add a constant term using `sm.add_constant()`.

In [ ]:
# Add constant and fit logistic regression
# TODO: Add a constant to X_train using sm.add_constant()
X_train_const = ...

# TODO: Fit a logistic regression model using sm.Logit()
model = sm.Logit(..., ...).fit()

# Display summary
print(model.summary())

In [ ]:
# Display coefficients and odds ratios
# TODO: Create a DataFrame with coefficients, odds ratios, and p-values
results = pd.DataFrame({
    'Coefficient': model.params,
    'Odds Ratio': ...,   # TODO: Calculate odds ratios using np.exp()
    'P-value': model.pvalues
})
print("\nCoefficients and Odds Ratios:")
print(results.round(4))

## Task #10: Model Evaluation

Make predictions on the test set using a threshold of 0.5. Calculate the confusion matrix and classification metrics (accuracy, precision, recall, F1 score).

In [ ]:
# Make predictions on test set
# TODO: Add constant to X_test and get predicted probabilities
X_test_const = sm.add_constant(X_test)
y_pred_prob = model.predict(X_test_const)

# TODO: Convert probabilities to binary predictions using threshold of 0.5
y_pred = ...

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
print("Confusion Matrix:")
print(f"                    Predicted")
print(f"                No Diabetes  Diabetes")
print(f"Actual No Diabetes   {cm[0,0]:6d}    {cm[0,1]:6d}")
print(f"       Diabetes      {cm[1,0]:6d}    {cm[1,1]:6d}")

# Classification metrics
# TODO: Calculate and print accuracy, precision, recall, and F1 score
print(f"\nClassification Metrics:")
print(f"Accuracy:  {accuracy_score(y_test, y_pred):.4f}")
print(f"Precision: {...:.4f}")
print(f"Recall:    {...:.4f}")
print(f"F1 Score:  {...:.4f}")

## Task #11: ROC Curve

Create a ROC curve and calculate the AUC (Area Under the Curve).

In [ ]:
# Calculate ROC curve
# TODO: Use roc_curve() to get false positive rate, true positive rate, and thresholds
fpr, tpr, thresholds = roc_curve(..., ...)

# TODO: Calculate AUC using roc_auc_score()
auc = ...

# Create the ROC plot
plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, 'b-', linewidth=2, label=f'ROC Curve (AUC = {auc:.3f})')
plt.plot([0, 1], [0, 1], 'r--', linewidth=1, label='Random Classifier')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate (Recall)')
plt.title('ROC Curve: Predicting Diabetes')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print(f"\nAUC = {auc:.3f}")

## Question #1 (4 points)

Based on the plots in Tasks #6 and #7

a) Describe the counts of `has_diabetes` specifiying whether there is a potential issue with class imbalance.

b) Is there evidence of a difference in `bmi` and `age` by diabetes status?

### Student Answer:

[Write your answer here]

---

Based on the logistic regression output from Task #9:

c) Interpret the odds ratios for `bmi` and `has_insurance`


### Student Answer:

[Write your answer here]

---

**To obtain full marks (4 points):**
- Correctly discuss bar plot (0.5 point)
- Correctly interpret box plots (1.5 point)
- Correctly interpret the 2 odds ratios (2 points)

## Question #2 (4 points)

Based on the classification metrics and ROC curve from Tasks #10 and #11:

a) What does the accuracy value tell us about the model's performance in identifying people who have diabetes?

b) What does the recall value tell us about the model's performance in identifying people who have diabetes?

c) What does the AUC value tell us about the model's discriminative ability?

d) If a clinic were using this model to screen patients for diabetes risk, would they be more concerned about false positives or false negatives? Explain why.

**To obtain full marks (4 points):**
- Correctly interpret the accuracy and recall metrics (1 point)
- Correctly interpret the AUC value (1 point)
- Identify whether false positives or false negatives are more concerning (1 point)
- Provide a clear justification in context (1 point)

### Student Answer:

[Write your answer here]